<a href="https://colab.research.google.com/github/Thashmikab/offshore-wind-layout-cable-tradeoff/blob/main/06_multiple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
%pip install -q py_wake==2.6.20 topfarm==2.6.2 pymoo==0.6.2

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 717.0/717.0 kB 28.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.2/347.2 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [38]:
import numpy as np
from pathlib import Path
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse.csgraph import minimum_spanning_tree
import numpy as np
import xarray as xr
from py_wake.site import XRSite
from py_wake.site.xrsite import GlobalWindAtlasSite
from py_wake.examples.data.dtu10mw import DTU10MW
from py_wake.literature.gaussian_models import Niayifar_PorteAgel_2016
from py_wake.wind_farm_models import PropagateDownwind
from py_wake.turbulence_models import CrespoHernandez
#from py_wake.wind_farm_models import EngineeringWindFarmModel
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.core.problem import Problem


In [63]:
# -------------------------------------------------------------
# Financial Cost Model: DTU 10MW on Jacket Foundation (UK £)
# -------------------------------------------------------------
N_TURBINES = 8

# Equipment Procurement CAPEX (per MW / per unit)
TURBINE_COST_PER_MW = 1_200_000   # Includes Nacelle + Rotor (£1.2M/MW)
TOWER_COST_BASE     = 940_000     # DTU 10MW 628t steel tower proxy
JACKET_COST_PER_MW  = 420_000     # Deeper water jacket structure (£420k/MW)

# Calculate fixed cost per node
single_turbine_cap = 10.0 # MW
cost_per_turbine_station = (
    (TURBINE_COST_PER_MW * single_turbine_cap) +
    TOWER_COST_BASE +
    (JACKET_COST_PER_MW * single_turbine_cap)
)

# Total array CAPEX baseline (8-turbine array = ~£137.1M asset cost)
array_station_capex = N_TURBINES * cost_per_turbine_station




# ------------------------------------------------------------------
# Cable Economic Model: Unit Costs (£ per Kilometer)
# ------------------------------------------------------------------
# 1. 66kV Inter-Array Metrics (Supply vs Total Installed)
COST_ARRAY_SUPPLY_PER_KM = 300_000
COST_ARRAY_INSTALLED_PER_KM = 750_000

# 2. Export Cable Metrics (Dependent on Offshore Distance)
# Close to shore (<100km): use 220kV HVAC proxy
COST_EXPORT_HVAC_PER_KM = 2_000_000

# Deep North Sea (>100km like Dogger Bank): use 320kV HVDC proxy
COST_EXPORT_HVDC_PER_KM = 3_200_000




TURBINE_DIAMETER = 178.3  # DTU 10MW Rotor Diameter (m)
MIN_SPACING = 4 * TURBINE_DIAMETER



wt = DTU10MW()
TI = 0.06



LEASE_X, LEASE_Y = 3000, 2000
D_REF_KM = 146.65                   # GWA Dogger Bank point, measured from the coast at Flamborough Head
D_MIN_KM, D_MAX_KM = 10, 150
POP_SIZE, N_GEN, SEED = 80, 100, 1

In [21]:
DATA = Path("data/dogger_bank_gwa.nc")

def load_dogger_bank_site():
    """Dogger Bank wind climate from the Global Wind Atlas (54.75 N, 1.90 E, 119 m, open sea).
    Downloaded once and cached in data/ so later runs don't depend on the GWA API."""
    if DATA.exists():
        return XRSite(xr.load_dataset(DATA))
    print("Downloading Dogger Bank wind climate from the Global Wind Atlas...")
    site = GlobalWindAtlasSite(lat=54.75, long=1.90, height=119, roughness=0.0002)
    DATA.parent.mkdir(exist_ok=True)
    site.ds.to_netcdf(DATA)
    return site


base_gwa_site = load_dogger_bank_site()

In [25]:
# 1. Load the CSV file once
transect_df = pd.read_csv("https://raw.githubusercontent.com/Thashmikab/offshore-wind-layout-cable-tradeoff/refs/heads/main/data/dogger_bank_transect.csv")

# Extract arrays for blazing-fast numpy operations
KNOWN_DISTANCES = transect_df['distance_km'].values
KNOWN_SCALES    = transect_df['U_rel'].values

def calculate_transect_scale_factor(distance_to_shore_km):
    """
    Empirically scales wind speed based on a Global Wind Atlas transect
    running from the Yorkshire coast out to Dogger Bank.
    """
    # Clip the incoming distance to ensure safe lookup bounds [0, 160]
    query_dist = np.clip(distance_to_shore_km, KNOWN_DISTANCES[0], KNOWN_DISTANCES[-1])

    # Linearly interpolate the scaling factor
    scale_factor = np.interp(query_dist, KNOWN_DISTANCES, KNOWN_SCALES)
    return float(scale_factor)


#print(calculate_transect_scale_factor(10))


In [34]:
from py_wake import turbulence_models
import inspect

for name, obj in inspect.getmembers(turbulence_models, inspect.isclass):
    print(f"- {name}")

- CrespoHernandez
- FrandsenWeight
- GCLTurbulence
- IECWeight
- RANSLUTTurbulence
- STF2005TurbulenceModel
- STF2017TurbulenceModel
- TurbulenceModel


In [36]:
# Instantiate the wake physics engine

wind_farm_model = Niayifar_PorteAgel_2016(
    site=base_gwa_site,
    windTurbines=wt,
    turbulenceModel=CrespoHernandez() # Matches the Niayifar 2016 publication standard
)

In [58]:
#@title Objective function


# Cable installed cost per km (£)
COST_ARRAY_KM = 750_000
COST_EXPORT_HVAC_KM = 2_000_000
COST_EXPORT_HVDC_KM = 3_200_000

N_WT = 8

def evaluate_wind_farm_layout(variables):
    """
    Objective function for NSGA-II 3-objective optimization.

    variables: list/array of 17 elements:
               [x1..x8, y1..y8, distance_to_shore]
    """
    # 1. Unpack variables
    x_coords = np.array(variables[0:N_WT])
    y_coords = np.array(variables[N_WT:2 * N_WT])
    dist_shore_km = variables[-1]

    coords = np.column_stack((x_coords, y_coords))
    # Compute the pairwise Euclidean distances
    # This returns a compressed, flat array of all distinct pairs
    distances = pdist(coords)




    # OBJECTIVE 1: Minimize (-AEP)
    scale_factor = calculate_transect_scale_factor(dist_shore_km)
    # Fix: Manually scale the wind speed data and create a new XRSite object
    scaled_ds = base_gwa_site.ds.copy(deep=True) # Create a copy to avoid modifying the original site
    if 'A' in scaled_ds:
        scaled_ds['A'] = base_gwa_site.ds['A'] * scale_factor
    else:
        scaled_ds['ws'] = base_gwa_site.ds['ws'] * scale_factor

    scaled_ds['TI'] = 0.06
    scaled_site = XRSite(scaled_ds)

    wind_farm_model.site = scaled_site



    x_offshore = x_coords + dist_shore_km * 1000          # move the lease box out to sea
    simulation_output = wind_farm_model(x_offshore, y_coords)
    aep_gwh = simulation_output.aep().sum().values
    obj_energy = -aep_gwh





    # -------------------------------------------------------------
    # OBJECTIVE 2: Minimize Inter-Array Cable Cost
    # -------------------------------------------------------------
    # Compute Minimum Spanning Tree of the turbine nodes
    dist_matrix = squareform(distances)
    mst_graph = minimum_spanning_tree(dist_matrix)      # calculates the path that connects all 8 turbines together such that the total length of the path is as short as possible, without creating any closed loops.
    array_cable_km = mst_graph.toarray().sum() / 1000.0 # Convert meters to km

    obj_array_cost = array_cable_km * COST_ARRAY_KM







    # -------------------------------------------------------------
    # OBJECTIVE 3: Minimize Export Cable Cost
    # -------------------------------------------------------------
    # Distance to shore acts directly as your export routing line
    export_cable_km = dist_shore_km




    # Constraint: closest pair must be at least MIN_SPACING apart (<= 0 means OK)
    spacing_violation = MIN_SPACING - distances.min()





    # Apply transmission style penalty based on commercial distance steps
    if export_cable_km > 100:
        obj_export_cost = export_cable_km * COST_EXPORT_HVDC_KM
    else:
        obj_export_cost = export_cable_km * COST_EXPORT_HVAC_KM

    return [obj_energy, obj_array_cost, obj_export_cost], spacing_violation

In [59]:
#@title Test the objective function on the starting grid

x0 = np.linspace(0, 2400, 4).repeat(2)
y0 = np.tile(np.linspace(0, 1600, 2), 4)

test_layout = np.concatenate([x0, y0, [50]])    # 4x2 grid, 50 km offshore
objectives, violation = evaluate_wind_farm_layout(test_layout)

print(f"AEP:          {-objectives[0]:.1f} GWh")
print(f"Array cost:   £{objectives[1] / 1e6:.1f}M")
print(f"Export cost:  £{objectives[2] / 1e6:.1f}M")
print(f"Spacing OK:   {violation <= 0}")

AEP:          380.3 GWh
Array cost:   £4.8M
Export cost:  £100.0M
Spacing OK:   True


In [ ]:
# Define Lease Area Boundaries (e.g., a 4km x 4km square box for the array)
X_MIN, X_MAX = 0.0, 4000.0  # in metres
Y_MIN, Y_MAX = 0.0, 4000.0  # in metres



# Define Distance to Shore Search Space
DIST_MIN, DIST_MAX = 10.0, 160.0  # in kilometres



class WindFarmOptimizationProblem(Problem):

    def __init__(self):
        # 17 variables: 8 x-coords, 8 y-coords, 1 distance-to-shore
        # 3 objectives: Maximize AEP (min -AEP), Min Array Cost, Min Export Cost
        super().__init__(n_var=17, n_obj=3, n_ieq_constr=0, vtype=float)



        # Define lower boundaries (xl) and upper boundaries (xu) for the algorithm
        self.xl = np.array([X_MIN]*8 + [Y_MIN]*8 + [DIST_MIN])
        self.xu = np.array([X_MAX]*8 + [Y_MAX]*8 + [DIST_MAX])





    def _evaluate(self, X, out, *args, **kwargs):
        """
        X: Matrix of shape (population_size, 17) containing candidate solutions.
        out: Dictionary where we store our computed objectives.
        """
        pop_size = X.shape[0]

        # Initialize objective arrays matching our population size
        f1_energy = np.zeros(pop_size)
        f2_array_cost = np.zeros(pop_size)
        f3_export_cost = np.zeros(pop_size)

        # Evaluate each individual configuration inside the current generation batch
        for i in range(pop_size):
            candidate_vars = X[i, :]

            # Execute your existing objective function pipeline
            obj_e, obj_a, obj_ex = evaluate_wind_farm_layout(candidate_vars)

            f1_energy[i] = obj_e
            f2_array_cost[i] = obj_a
            f3_export_cost[i] = obj_ex

        # Pack computed matrices back into pymoo's output structure 'F'
        out["F"] = np.column_stack([f1_energy, f2_array_cost, f3_export_cost])


In [61]:
#@title NSGA-II problem

class WindFarmProblem(ElementwiseProblem):

    def __init__(self):
        lower = [0] * N_WT + [0] * N_WT + [D_MIN_KM]              # x, y, distance
        upper = [LEASE_X] * N_WT + [LEASE_Y] * N_WT + [D_MAX_KM]
        super().__init__(n_var=2 * N_WT + 1,   # 17 variables
                         n_obj=3,              # energy, array cost, export cost
                         n_ieq_constr=1,       # spacing
                         xl=lower, xu=upper)

    def _evaluate(self, v, out, *args, **kwargs):
        objectives, spacing_violation = evaluate_wind_farm_layout(v)
        out["F"] = objectives
        out["G"] = [spacing_violation]

In [64]:
#@title Run NSGA-II

problem = WindFarmProblem()

# Starting population: random layouts, plus the 4x2 grid so at least one is feasible
rng = np.random.default_rng(SEED)
X0 = rng.uniform(problem.xl, problem.xu, size=(POP_SIZE, problem.n_var))
X0[0] = np.concatenate([x0, y0, [D_REF_KM]])

result = minimize(problem,
                  NSGA2(pop_size=POP_SIZE, sampling=X0),
                  ("n_gen", N_GEN),
                  seed=SEED,
                  verbose=True)        # prints progress each generation

front = pd.DataFrame({
    "Distance_to_shore_km": result.X[:, -1],
    "AEP_GWh":              -result.F[:, 0],
    "Array_cost_GBP_M":     result.F[:, 1] / 1e6,
    "Export_cost_GBP_M":    result.F[:, 2] / 1e6,
}).sort_values("Distance_to_shore_km").reset_index(drop=True)

print(f"{len(front)} trade-off layouts found")
print(front.round(1).to_string())

n_gen  |  n_eval  | n_nds  |     cv_min    |     cv_avg    |      eps      |   indicator  
     1 |       80 |      1 |  0.000000E+00 |  4.564517E+02 |             - |             -
     2 |      160 |      2 |  0.000000E+00 |  3.562833E+02 |  1.0000000000 |         ideal
     3 |      240 |      3 |  0.000000E+00 |  2.979176E+02 |  0.8613746949 |         nadir
     4 |      320 |      4 |  0.000000E+00 |  2.499027E+02 |  0.3527682677 |         ideal
     5 |      400 |      5 |  0.000000E+00 |  2.083846E+02 |  0.0159897525 |             f
     6 |      480 |      6 |  0.000000E+00 |  1.768282E+02 |  0.7218859109 |         ideal
     7 |      560 |      8 |  0.000000E+00 |  1.335414E+02 |  0.1413158933 |         ideal
     8 |      640 |     11 |  0.000000E+00 |  7.463316E+01 |  0.1023894012 |         nadir
     9 |      720 |     16 |  0.000000E+00 |  1.963212E+01 |  0.0683912649 |             f
    10 |      800 |     27 |  0.000000E+00 |  0.000000E+00 |  0.0058163906 |         ideal

In [65]:
#@title 3D Pareto front (interactive)
import plotly.express as px
import plotly.graph_objects as go

# Where the starting 4x2 grid at Dogger Bank sits, for comparison
grid_obj, _ = evaluate_wind_farm_layout(np.concatenate([x0, y0, [D_REF_KM]]))

fig = px.scatter_3d(
    front,
    x="Array_cost_GBP_M", y="Export_cost_GBP_M", z="AEP_GWh",
    color="Distance_to_shore_km", color_continuous_scale="Viridis",
    hover_data={"Distance_to_shore_km": ":.1f", "AEP_GWh": ":.1f",
                "Array_cost_GBP_M": ":.2f", "Export_cost_GBP_M": ":.1f"},
    labels={"Array_cost_GBP_M": "Array cable cost (£M)",
            "Export_cost_GBP_M": "Export cable cost (£M)",
            "AEP_GWh": "AEP (GWh)",
            "Distance_to_shore_km": "Distance to shore (km)"},
    title="Pareto front: energy vs array and export cable cost",
)
fig.update_traces(marker=dict(size=5))

fig.add_trace(go.Scatter3d(
    x=[grid_obj[1] / 1e6], y=[grid_obj[2] / 1e6], z=[-grid_obj[0]],
    mode="markers", marker=dict(size=8, color="red", symbol="x"),
    name="4x2 grid at Dogger Bank"))

fig.update_layout(height=650, legend=dict(x=0, y=1))
fig.show()